In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils.DataLoader import DataLoader

from transformers import BertTokenizer, BertModel
from tqdm import tqdm
import torch


In [2]:
# dataloader = DataLoader(r'.\ctg-studies.json')
# df = dataloader.load_data()
# # df.head()
# df.to_csv(r'.\ctg-studies.csv', index=False)

In [3]:
df = pd.read_csv(r'.\ctg-studies.csv')
nctid_eligibility = df[['NCT ID', 'Eligibility Criteria']]
nctid_eligibility.head()

,NCT ID,Eligibility Criteria
0,NCT00019136,DISEASE CHARACTERISTICS:\r\n\r\n* Histological...
1,NCT00085930,Inclusion Criteria:\r\n\r\nHigh risk neuroblas...
2,NCT00881920,INCLUSION CRITERIA:\r\n\r\nBLOOD PROCUREMENT:\...
3,NCT00902044,INCLUSION CRITERIA:\r\n\r\nProcurement Eligibi...
4,NCT00924326,* INCLUSION CRITERIA:\r\n\r\n 1. Patient must...


In [5]:
tokenizer = BertTokenizer.from_pretrained("microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract")
model = BertModel.from_pretrained("microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract")


In [9]:
# Tokenize and get embeddings for each eligibility criteria
embeddings = []
for text in tqdm(nctid_eligibility['Eligibility Criteria']):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    outputs = model(**inputs)
    pooled_output = torch.mean(outputs.last_hidden_state, dim=1)
    embeddings.append(pooled_output.detach().numpy())


# Flatten the embeddings
feature_matrix = [emb.flatten() for emb in embeddings]
pd.DataFrame(feature_matrix).to_csv(r'.\feature-matrix.csv', index=False)

  0%|          | 0/1267 [00:00<?, ?it/s]

 11%|█▏        | 145/1267 [15:20<2:24:39,  7.74s/it]

In [ ]:
from sklearn.cluster import KMeans

# Specify the number of clusters
num_clusters = 5  # You can adjust this based on your task

# Perform k-means clustering
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
cluster_labels = kmeans.fit_predict(feature_matrix)

# Add cluster labels to the original DataFrame
nctid_eligibility['Cluster_Labels'] = cluster_labels

# Display the clusters
print("Cluster Labels:")
print(nctid_eligibility['Cluster_Labels'].value_counts())

# Visualize the clusters
sns.countplot(x='Cluster_Labels', data=nctid_eligibility)
plt.title("Distribution of Clusters")
plt.show()